# ABIDES Declarative Configuration System — Tutorial

This notebook demonstrates the **declarative configuration system** for ABIDES market simulations.  
Instead of writing procedural `build_config()` functions, you can:

- Build simulations from **composable templates**
- **Enable/disable agent types** with a fluent API
- Set **per-agent computation delays**
- **Serialize** configs as YAML or JSON
- **Register custom agents** with the plugin registry
- Use the **AI discoverability API** for programmatic introspection

📖 Full reference: [`docs/reference/config-system.md`](../docs/reference/config-system.md)

In [ ]:
import pathlib
import sys

# Add ABIDES packages to path (assumes notebook is in <workspace>/notebooks/)
_ws = pathlib.Path().resolve()
if _ws.name == "notebooks":
    _ws = _ws.parent
for _pkg in ("abides-core", "abides-markets"):
    _p = str(_ws / _pkg)
    if _p not in sys.path:
        sys.path.insert(0, _p)

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from abides_core import abides
from abides_core.utils import fmt_ts, ns_date, str_to_ns
from abides_markets.config_system import (
    BaseAgentConfig,
    SimulationBuilder,
    compile,
    config_to_dict,
    get_config_schema,
    list_agent_types,
    list_templates,
    load_config,
    register_agent,
    registry,
    save_config,
    validate_config,
)

---
## 1. Quick Start — Build, Compile, Run

The three-step workflow:
1. **Build** a `SimulationConfig` using the `SimulationBuilder`
2. **Compile** it into the runtime dict the `Kernel` expects
3. **Run** the simulation with `abides.run()`

> **Tip:** For most use-cases, prefer `run_simulation(config)` from
> `abides_markets.simulation` — it compiles a fresh runtime dict internally
> and returns a typed, immutable `SimulationResult`.  The manual
> `compile()` → `abides.run()` pattern shown here is the low-level path;
> each runtime dict is **consumed once** (agents accumulate state during the run).

In [ ]:
# Build a config from the rmsc04 template
config = (
    SimulationBuilder()
    .from_template("rmsc04")
    .market(ticker="ABM")
    .seed(42)
    .build()
)

print(f"Ticker: {config.market.ticker}")
print(f"Date: {config.market.date}")
print(f"Trading hours: {config.market.start_time} — {config.market.end_time}")
print(f"Seed: {config.simulation.seed}")
print(f"Agent groups: {list(config.agents.keys())}")

In [ ]:
# Compile to a runtime dict
runtime = compile(config)

print(f"Runtime dict keys: {list(runtime.keys())}")
print(f"Total agents: {len(runtime['agents'])}")
print(f"Default computation delay: {runtime['default_computation_delay']} ns")

In [ ]:
# Run the simulation
end_state = abides.run(runtime)

print(f"Simulation completed. Agents in end_state: {len(end_state['agents'])}")

---
## 2. Templates — Composable Starting Points

Templates provide sensible defaults for common simulation scenarios.  
**Base templates** define a full configuration; **overlay templates** add agent groups on top.

In [ ]:
# List all available templates
templates = list_templates()
for t in templates:
    overlay = " (overlay)" if t["is_overlay"] else ""
    print(f"  {t['name']}{overlay}: {t['description']}")

### Template stacking

You can stack multiple templates — later ones override earlier ones.  
Here we start with `rmsc04` and add a POV execution agent via the `with_execution` overlay:

In [ ]:
config_stacked = (
    SimulationBuilder()
    .from_template("rmsc04")
    .from_template("with_execution")  # adds a POV execution agent
    .seed(42)
    .build()
)

print("Agent groups after stacking:")
for name, group in config_stacked.agents.items():
    status = "enabled" if group.enabled else "disabled"
    print(f"  {name}: {group.count} agents ({status})")

---
## 3. Builder API — Fine-Grained Control

The fluent builder lets you override any parameter:

In [ ]:
config_custom = (
    SimulationBuilder()
    .from_template("rmsc04")
    .market(ticker="AAPL", date="20220315", end_time="11:00:00")
    .oracle(r_bar=150_000)                        # $1500.00 fundamental
    .exchange(book_log_depth=20)
    .enable_agent("noise", count=500)              # fewer noise agents
    .enable_agent("value", count=50, r_bar=150_000)
    .disable_agent("momentum")                     # no momentum agents
    .computation_delay(100)                        # global default: 100ns
    .seed(123)
    .log_orders(True)
    .build()
)

print(f"Ticker: {config_custom.market.ticker}")
print(f"Oracle r_bar: ${config_custom.market.oracle.r_bar / 100:.2f}")
print(f"Noise agents: {config_custom.agents['noise'].count}")
print(f"Value agents: {config_custom.agents['value'].count}")
print(f"Momentum enabled: {config_custom.agents['momentum'].enabled}")
print(f"Default computation delay: {config_custom.infrastructure.default_computation_delay} ns")

---
## 4. Per-Agent Computation Delays

Different agent types can have different "thinking times".  
Market makers might be fast (low delay) while background noise agents are slow (high delay).  
The `computation_delay` parameter overrides the simulation-level default **per agent group**.

In [ ]:
from collections import Counter

config_delays = (
    SimulationBuilder()
    .from_template("rmsc04")
    .computation_delay(50)                                       # global default
    .enable_agent("adaptive_market_maker", count=2,
                  computation_delay=10)                          # fast MM
    .enable_agent("noise", count=1000, computation_delay=500)   # slow noise
    .enable_agent("value", count=102)                            # uses global default (50ns)
    .seed(42)
    .build()
)

runtime_delays = compile(config_delays)

# Show the per-agent overrides
per_agent = runtime_delays.get("per_agent_computation_delays", {})
print(f"Global default: {runtime_delays['default_computation_delay']} ns")
print(f"Agents with custom delays: {len(per_agent)}")

# Summarize by agent type
delay_values = Counter(per_agent.values())
for delay, count in sorted(delay_values.items()):
    print(f"  {count} agents with {delay} ns delay")

You can also use the convenience method `agent_computation_delay()` on the builder:

In [ ]:
# Equivalent: set computation delay for a specific agent type
config_alt = (
    SimulationBuilder()
    .from_template("rmsc04")
    .agent_computation_delay("noise", 500)
    .agent_computation_delay("adaptive_market_maker", 10)
    .seed(42)
    .build()
)

print(f"Noise delay: {config_alt.agents['noise'].params.get('computation_delay')} ns")
print(f"MM delay: {config_alt.agents['adaptive_market_maker'].params.get('computation_delay')} ns")

---
## 5. Agent Registry — Discover & Extend

All built-in agent types are self-registered. You can discover them programmatically.

In [ ]:
# List all registered agent types
agent_types = list_agent_types()

print(f"{'Name':<25} {'Category':<15} {'Description'}")
print("-" * 80)
for a in agent_types:
    print(f"{a['name']:<25} {a['category']:<15} {a.get('description', '')}")

In [ ]:
# Inspect parameters for a specific agent type
schema = registry.get_json_schema("value")
print("ValueAgent parameters:")
for name, prop in schema["properties"].items():
    desc = prop.get("description", "")
    default = prop.get("default", "—")
    print(f"  {name}: default={default}  ({desc})")

### Registering a custom agent

Third-party agents can be registered with `@register_agent` and used in configs just like built-in ones.  
The config model must extend `BaseAgentConfig` and implement `create_agents()`.

In [ ]:
from pydantic import Field

_CUSTOM_DESC = "Custom noise agent with configurable threshold"


@register_agent("my_custom_noise", category="background", description=_CUSTOM_DESC)
class MyCustomNoiseConfig(BaseAgentConfig):
    """Custom noise agent that wraps the built-in NoiseAgent."""

    threshold: float = Field(default=0.05, description="Custom threshold.")

    def create_agents(self, count, id_start, master_rng, context):
        from abides_core.utils import get_wake_time, str_to_ns
        from abides_markets.agents import NoiseAgent

        log = self.log_orders if self.log_orders is not None else context.log_orders
        noise_mkt_open = context.mkt_open + str_to_ns("-00:30:00")
        date_ns = context.mkt_open - str_to_ns("09:30:00")
        noise_mkt_close = date_ns + str_to_ns("16:00:00")

        agents = []
        for j in range(id_start, id_start + count):
            rng = np.random.RandomState(
                seed=master_rng.randint(low=0, high=2**32, dtype="uint64")
            )
            agents.append(
                NoiseAgent(
                    id=j,
                    name=f"MyCustomNoise_{j}",
                    type="MyCustomNoiseAgent",
                    symbol=context.ticker,
                    starting_cash=self.starting_cash,
                    wakeup_time=get_wake_time(
                        noise_mkt_open, noise_mkt_close, rng
                    ),
                    log_orders=log,
                    random_state=rng,
                )
            )
        return agents


print("Custom agent registered!")
print(f"Registered types: {registry.registered_names()}")

In [ ]:
# Use the custom agent in a simulation
config_custom_agent = (
    SimulationBuilder()
    .from_template("rmsc04")
    .enable_agent("my_custom_noise", count=10, threshold=0.1)
    .seed(42)
    .build()
)

runtime_custom = compile(config_custom_agent)
print(f"Total agents (1117 rmsc04 + 10 custom): {len(runtime_custom['agents'])}")

---
## 6. Serialization — YAML & JSON

Configs can be saved to YAML or JSON and loaded back. This is useful for:
- Sharing experiment configurations
- Version-controlling simulation setups
- Parameter sweeps from config files

In [ ]:
# Save to YAML
config_to_save = (
    SimulationBuilder()
    .from_template("rmsc04")
    .market(ticker="AAPL")
    .enable_agent("noise", count=500, computation_delay=200)
    .seed(42)
    .build()
)

yaml_path = Path(tempfile.gettempdir()) / "abides_config_demo.yaml"
save_config(config_to_save, yaml_path)

# Display the YAML content
print(yaml_path.read_text())

In [ ]:
# Load it back and verify
loaded = load_config(yaml_path)

print(f"Loaded ticker: {loaded.market.ticker}")
print(f"Loaded noise agent count: {loaded.agents['noise'].count}")
print(f"Loaded noise computation_delay: {loaded.agents['noise'].params.get('computation_delay')}")
print(f"Loaded seed: {loaded.simulation.seed}")

In [ ]:
# JSON works too
json_path = Path(tempfile.gettempdir()) / "abides_config_demo.json"
save_config(config_to_save, json_path)

d = config_to_dict(config_to_save)
print(json.dumps(d, indent=2)[:500], "...")

---
## 7. AI Discoverability API

The config system exposes structured introspection functions.  
These are designed for LLM tool-calling but also useful interactively.

In [ ]:
# Full JSON Schema for the config structure
schema = get_config_schema()
print(f"Top-level config sections: {list(schema['properties'].keys())}")
print(f"Schema keys: {list(schema.keys())}")

In [ ]:
# Validate a config dict (returns structured result)
print("Valid config:", validate_config({"simulation": {"seed": 42}}))
print("Invalid config:", validate_config({"agents": {"noise": {"count": -1}}}))

---
## 8. Running a Simulation and Extracting Results

The compiled runtime dict is fully compatible with the existing ABIDES infrastructure.  
Below we use the low-level `compile()` → `abides.run()` path to access raw agent
state for plotting.  For a higher-level API, see `run_simulation()` in
`abides_markets.simulation`.

In [ ]:
# Build and run a simulation
sim_config = (
    SimulationBuilder()
    .from_template("rmsc04")
    .market(end_time="10:30:00")  # 1 hour of trading
    .seed(42)
    .build()
)

sim_runtime = compile(sim_config)
sim_end_state = abides.run(sim_runtime)

In [ ]:
# Extract L1 order book data
order_book = sim_end_state["agents"][0].order_books["ABM"]
L1 = order_book.get_L1_snapshots()

best_bids = pd.DataFrame(L1["best_bids"], columns=["time", "price", "qty"])
best_asks = pd.DataFrame(L1["best_asks"], columns=["time", "price", "qty"])

# Convert time to seconds from midnight for plotting
best_bids["time"] = best_bids["time"].apply(lambda x: x - ns_date(x))
best_asks["time"] = best_asks["time"].apply(lambda x: x - ns_date(x))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(best_bids.time, best_bids.price, label="Best Bid", alpha=0.8)
ax.plot(best_asks.time, best_asks.price, label="Best Ask", alpha=0.8)

band = 150
ax.set_ylim(100_000 - band, 100_000 + band)
ax.set_ylabel("Price (cents)")
ax.set_title("L1 Best Bid / Ask — Config System Simulation")
ax.legend()

time_mesh = np.arange(str_to_ns("09:30:00"), str_to_ns("10:40:00"), 1e9 * 60 * 10)
ax.set_xticks(time_mesh)
ax.set_xticklabels([fmt_ts(int(t)).split(" ")[1] for t in time_mesh], rotation=45)
plt.tight_layout()
plt.show()

---
## 9. Comparing Old vs. New Config System

The new system produces the exact same runtime dict as `rmsc04.build_config()`.  
Both approaches are fully supported — use whichever fits your workflow.

In [ ]:
from collections import Counter

from abides_markets.configs import rmsc04

# Old procedural approach
old_config = rmsc04.build_config(seed=42)
old_types = Counter(type(a).__name__ for a in old_config["agents"])

# New declarative approach
new_config = SimulationBuilder().from_template("rmsc04").seed(42).build()
new_runtime = compile(new_config)
new_types = Counter(type(a).__name__ for a in new_runtime["agents"])

print(f"Old system agent count: {len(old_config['agents'])}")
print(f"New system agent count: {len(new_runtime['agents'])}")
print(f"Agent types match: {old_types == new_types}")
print(f"Runtime dict keys match: {set(old_config.keys()).issubset(set(new_runtime.keys()))}")
print()
print("Agent composition:")
for agent_type, count in sorted(new_types.items()):
    print(f"  {agent_type}: {count}")

---
## 10. Summary

| Feature | How |
|---------|-----|
| Start from template | `SimulationBuilder().from_template("rmsc04")` |
| Override market params | `.market(ticker="AAPL", date="20220315")` |
| Enable/disable agents | `.enable_agent("noise", count=500)` / `.disable_agent("momentum")` |
| Per-agent delays | `.enable_agent("noise", count=500, computation_delay=200)` |
| Stack templates | `.from_template("rmsc04").from_template("with_execution")` |
| Serialize to YAML | `save_config(config, "sim.yaml")` |
| Register custom agent | `@register_agent("my_agent", category="strategy")` |
| AI introspection | `list_agent_types()`, `get_config_schema()` |
| Run (recommended) | `run_simulation(config)` → `SimulationResult` |
| Run (low-level) | `runtime = compile(config); abides.run(runtime)` |